# Session 2 · Cleansing

**Goal:** load the file the right way, then fix whitespace, codes that mean "unknown," casing, names, keys, and multi-value fields. Save the result in a format that keeps what we fixed.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 2

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import pandas as pd
from lnl import check, cleaning
from lnl.check import ___

## 1. Load it the right way
Three settings make the difference:
- `dtype=str` reads **every** column as text. Nothing gets "helpfully" converted, so TDCJ and SID numbers keep their leading zeros.
- `keep_default_na=False` and `na_filter=False` stop pandas from deciding what counts as missing. By default it treats `NA`, `N/A`, `null`, `None` and more as missing, which is a decision that belongs to us and the data owner.

In [ ]:
df = pd.read_csv(PATHS["data"], dtype=str, keep_default_na=False, na_filter=False)

print("TDCJ #02381457 exists:", (df["tdcj_number"] == "02381457").any())    # Q5 is fixed
df[["tdcj_number", "sid_number"]].head()

## 2. Strip and null
Last session's fix, now saved as a reusable function (`cleaning.strip_and_null`) so it's applied exactly the same way every time.

In [ ]:
df = cleaning.strip_and_null(df)
df.isna().sum().to_frame("missing values")

## 3. Make `tdcj_number` the record ID
Every record has exactly one TDCJ number: 8 digits, leading zeros included, never repeated. That makes it the record's **ID**, and from here on we treat it as one:

- `cleaning.set_record_id` first **proves** it's a real ID: none missing, all 8 digits, none repeated. If any record fails, it stops with an error instead of guessing.
- It then makes `tdcj_number` the DataFrame's **index**, so every table from now on is labeled by TDCJ number instead of an anonymous row number, and you can look a record up directly.

We stripped whitespace first on purpose: an ID with a stray space in front would fail the check, not slip through.

In [ ]:
df = cleaning.set_record_id(df)

print("Index:", df.index.name, "| unique:", df.index.is_unique)
print("TDCJ #02381457 exists:", "02381457" in df.index)       # Q5 is fixed
df.loc["02381457", ["last_name", "first_name", "alias", "sid_number"]]

## 4. Codes that mean "unknown"
Source systems often put a placeholder in a field instead of leaving it blank. Let's look for the usual suspects:

In [ ]:
suspects = ["N/A", "U", "UNK", "99:99", "NONE"]
for col in ["disciplinary_points", "escape_risk", "protective_custody", "intake_time", "housing_restriction"]:
    counts = df[col].value_counts()
    print(f"{col:22} {counts[counts.index.isin(suspects)].to_dict()}")

Each one needs a decision, and **the data owner makes it, not the analyst**:

| Column | Code | What it means | Decision |
|---|---|---|---|
| `disciplinary_points` | `N/A` | not recorded | make it null |
| `escape_risk` | `U`, `UNK` | unknown | make it null |
| `protective_custody` | `U` | unknown | make it null |
| `intake_time` | `99:99` | the old system's "unknown time" | make it null |
| `housing_restriction` | `NONE` | **no restriction is needed** | **keep it.** It's a real answer |

The last row is the trap. `NONE` *looks* like a missing value, but it's an answer: this person needs no special housing. A blank means "we don't know." Treat them the same and every housing report is wrong.

Single-character flags raise one more question for the data owner: **does a blank flag mean No** (an unchecked box) **or unknown?** In this data it means unknown. Ask; don't assume.

The decisions live in a data dictionary, so they're applied identically every time:

In [ ]:
cleaning.NULL_TOKENS

In [ ]:
before = df[list(cleaning.NULL_TOKENS)].isna().sum()
df = cleaning.apply_null_tokens(df)
after = df[list(cleaning.NULL_TOKENS)].isna().sum()
pd.DataFrame({"missing before": before, "missing after": after})

🧪 **Your turn #1.** How many `housing_restriction` values are the text `NONE`?

In [ ]:
answer_1 = ___      # compare df["housing_restriction"] to "NONE", then add up the matches with .sum()
check.s2_ex1(answer_1)

## 5. Casing and names

In [ ]:
df.loc[df["last_name"].str.upper() == "KENT", ["last_name", "first_name", "alias", "sid_number"]]    # labeled by TDCJ number

Three records, each with its own TDCJ number: the same person, three spellings, and one record missing its SID. For **matching**, we add standardized columns (uppercase, single spaces). The originals stay for **display**.

In [ ]:
df = cleaning.standardize_names(df)
print("Records with last name KENT:", (df["last_name_std"] == "KENT").sum())    # Q3 is fixed

Proper case for reports is harder than it looks. `str.title()`, the obvious tool, gets a lot of names wrong:

In [ ]:
examples = pd.Series(["MCDONALD", "O'CONNOR", "DE LA CRUZ", "WORTHINGTON III", "T'CHALLA", "SMITH JR", "VAN DYNE"])
pd.DataFrame({"raw": examples,
              "str.title()": examples.str.title(),
              "cleaning.display_name()": examples.map(cleaning.display_name)})

`display_name()` uses rules for Mc, apostrophes, particles like *de la*, and suffixes. No rule set is perfect (think *MacDonald* vs *Macy*), which is why **matching always uses the `_std` columns**, never the display version.

🧪 **Your turn #2.** How many different spellings of MCDONALD are in the original `last_name` column?

In [ ]:
answer_2 = ___      # filter rows where last_name_std == "MCDONALD", then .nunique() on last_name
check.s2_ex2(answer_2)

## 6. Check the keys
Before trusting any join or count, check the identifiers. In this system a person keeps the same **SID** for life but gets a **new TDCJ number each time they're incarcerated**. So a TDCJ number identifies a *record*, and a SID identifies a *person*.

In [ ]:
cleaning.key_checks(df)

What that tells us:
- TDCJ numbers are clean: none missing, none duplicated, all 8 digits (`set_record_id` would have stopped otherwise).
- About 3,000 records have no SID, so they can't be linked to a person.
- No SID has conflicting last names (good; that would signal a bad merge).
- Thousands of people have more than one record, so **counting rows is not counting people**.

🧪 **Your turn #3.** How many people have more than one record?

In [ ]:
records_per_person = df.groupby("sid_number").size()
answer_3 = ___      # how many values in records_per_person are greater than 1?
check.s2_ex3(answer_3)

## 7. Several values in one cell
`superpower` packs several values into one cell (`FLIGHT;SUPER STRENGTH`). You can't count or filter by power properly until each one gets its own row. That's called *exploding* the column.

In [ ]:
powers = cleaning.explode_powers(df)
print(f"{len(df):,} records became {len(powers):,} (record, power) rows")
powers["power"].value_counts().head(10)

In [ ]:
print("contains 'FLIGHT':", df["superpower"].str.contains("FLIGHT", na=False).sum())
print("exactly 'FLIGHT': ", (powers["power"] == "FLIGHT").sum())

`contains` also matches `FLIGHT (WINGS)` and `FLIGHT (JETPACK)`. That may or may not be what the question means; exploded values let you choose deliberately.

## 8. Save it: Parquet, not CSV

In [ ]:
out = PATHS["dir"]
df.to_csv(out / "session2_clean.csv")          # the index (tdcj_number) is written too
df.to_parquet(out / "session2_clean.parquet")  # and comes back as the index

from_csv = pd.read_csv(out / "session2_clean.csv", dtype=str, keep_default_na=False)
from_parquet = pd.read_parquet(out / "session2_clean.parquet")
print("ID survives the Parquet round trip:", from_parquet.index.name, from_parquet.index[0])
pd.DataFrame({"missing values": [df.isna().sum().sum(), from_csv.isna().sum().sum(), from_parquet.isna().sum().sum()]},
             index=["in memory", "reloaded from CSV", "reloaded from Parquet"])

A CSV can't tell "missing" apart from "empty text": both are written as nothing between two commas. Reload it carefully and **every null we just created is gone**. Parquet keeps nulls exactly, and after next session it keeps real dates too. Use Parquet (or a Delta table) between steps, and CSV only for handing files to people. Either way, never save with `index=False` now: the index is the TDCJ number.

## Trust Test check-in

In [ ]:
pd.Series({
    "Q1 Records missing a custody level": df["custody_level"].isna().sum(),
    "Q2 Distinct gender values":          df["gender"].nunique(dropna=False),
    "Q3 Records with last name KENT":     (df["last_name_std"] == "KENT").sum(),
    "Q4 Earliest projected release date": df["projected_release_date"].dropna().min(),
    "Q5 Does TDCJ #02381457 exist?":      "02381457" in df.index,
}, name="answer").to_frame()

Four out of five are right now. Q4 is still text sorted alphabetically.

## Wrap-up
- Load untrusted files **as text**, and decide types yourself.
- Codes that mean "unknown" go in a **data dictionary** agreed with the data owner. `NONE` is not always null.
- **Match** on standardized values; **display** the originals.
- `tdcj_number` is the **record ID**: prove it (present, 8 digits, unique), then make it the index. Rows aren't people; SIDs are.
- Save with **Parquet**, not CSV.

**Next session:** nine date formats, two-digit years, and dates that parse fine but can't possibly be true.

---
## Answer key
```python
answer_1 = (df["housing_restriction"] == "NONE").sum()
answer_2 = df.loc[df["last_name_std"] == "MCDONALD", "last_name"].nunique()
answer_3 = (records_per_person > 1).sum()
```